In [7]:
import pandas as pd

In [8]:
players = pd.read_csv('data/Player.csv')
matches = pd.read_csv('data/Match.csv')
characters = pd.read_csv('data/Character.csv')
players['name'] = players['name'].replace('', pd.NA).fillna(players['discordName'])


In [9]:
PLAYER_ID = "1050952903751381012"

In [15]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Resolve internal player ID from Discord ID
player_row = players[players['discordId'] == int(PLAYER_ID)]
player_id = int(player_row['id'].values[0])
player_label = player_row['name'].values[0] if ('name' in players.columns and pd.notna(player_row['name'].values[0]) and str(player_row['name'].values[0]).strip() != '') else str(player_id)
print(f"Analyzing Player: {player_label}  (ID: {player_id}, Discord: {PLAYER_ID})")

char_map = characters.set_index('id')['name'].to_dict()

# Build unified per-match view
as_winner = matches[matches['winnerId'] == player_id].copy()
as_winner['result'] = 'Win'
as_winner['player_char'] = as_winner['winnerCharacterId']
as_winner['opp_id'] = as_winner['loserId']
as_winner['opp_char'] = as_winner['loserCharacterId']

as_loser = matches[matches['loserId'] == player_id].copy()
as_loser['result'] = 'Loss'
as_loser['player_char'] = as_loser['loserCharacterId']
as_loser['opp_id'] = as_loser['winnerId']
as_loser['opp_char'] = as_loser['winnerCharacterId']

pm = pd.concat([as_winner, as_loser], ignore_index=True)
pm['season'] = 'S' + pm['ladderResetId'].astype(str)
pm['player_char_name'] = pm['player_char'].map(char_map)
pm['opp_char_name'] = pm['opp_char'].map(char_map)
# Player label: use name if available, else "P{id}"
player_label_map = {
    row['id']: row['name'] if ('name' in players.columns and pd.notna(row.get('name')) and str(row.get('name', '')).strip() != '') else str(row['id'])
    for _, row in players.iterrows()
}
pm['opp_label'] = pm['opp_id'].map(player_label_map)

def matchup_stats(df, group_col='opp_label', top_n=None):
    stats = df.groupby(group_col).agg(
        Games=('result', 'count'),
        Wins=('result', lambda x: (x == 'Win').sum()),
        Losses=('result', lambda x: (x == 'Loss').sum()),
    ).reset_index()
    stats['WinRate'] = (stats['Wins'] / stats['Games'] * 100).round(1)
    stats = stats.sort_values('Games', ascending=False)
    if top_n:
        stats = stats.head(top_n)

    return stats

seasons = sorted(pm['season'].unique())
print(f"Total matches: {len(pm)}  |  Seasons: {seasons}")

Analyzing Player: berries_2993  (ID: 71, Discord: 1050952903751381012)
Total matches: 642  |  Seasons: ['S10', 'S14', 'S18', 'S22', 'S24', 'S25', 'S26', 'S4', 'S7']


In [12]:
# ── OVERALL MATCHUPS ──────────────────────────────────────────────────────────
overall = matchup_stats(pm)

# Top 10 opponents by games played (stacked bar)
top10 = overall.head(20)
long10 = top10.melt(id_vars='opp_label', value_vars=['Wins', 'Losses'],
                    var_name='Result', value_name='Count')
fig1 = px.bar(
    long10, x='opp_label', y='Count', color='Result',
    color_discrete_map={'Wins': '#2ecc71', 'Losses': '#e74c3c'},
    title=f'Top 20 Opponents by Games Played — All Seasons ({player_label})',
    labels={'opp_label': 'Opponent', 'Count': 'Games'},
    text_auto=True, barmode='stack',
    category_orders={'opp_label': top10['opp_label'].tolist()},
)
fig1.update_layout(legend_title_text='Result')
fig1.show()

# Win rate vs all opponents with ≥3 games (bubble chart: size = games)
min_games = 3
wr_df = overall[overall['Games'] >= min_games].sort_values('WinRate', ascending=True).reset_index(drop=True)
fig2 = px.scatter(
    wr_df, x='WinRate', y='opp_label', size='Games', color='WinRate',
    color_continuous_scale='RdYlGn', range_color=[0, 100],
    title=f'Win Rate vs Each Opponent (≥{min_games} games) — All Seasons ({player_label})',
    labels={'opp_label': 'Opponent', 'WinRate': 'Win Rate (%)'},
    hover_data={'Games': True, 'Wins': True, 'Losses': True, 'WinRate': True},
)
fig2.add_vline(x=50, line_dash='dash', line_color='gray', annotation_text='50%')
fig2.update_layout(
    coloraxis_showscale=False,
    height=max(420, len(wr_df) * 28 + 120),
    yaxis={'categoryorder': 'array', 'categoryarray': wr_df['opp_label'].tolist()},
)
fig2.show()


In [13]:
# ── PER-SEASON BREAKDOWN ──────────────────────────────────────────────────────

# Season overview: total W/L/winrate per season
season_overview = pm.groupby('season').agg(
    Games=('result', 'count'),
    Wins=('result', lambda x: (x == 'Win').sum()),
    Losses=('result', lambda x: (x == 'Loss').sum()),
).reset_index()
season_overview['WinRate'] = (season_overview['Wins'] / season_overview['Games'] * 100).round(1)

seasons_sorted = sorted(seasons, key=lambda s: int(s[1:]))

fig_sov = make_subplots(specs=[[{"secondary_y": True}]])
fig_sov.add_trace(go.Bar(
    x=season_overview['season'], y=season_overview['Wins'],
    name='Wins', marker_color='#2ecc71', text=season_overview['Wins'], textposition='inside',
), secondary_y=False)
fig_sov.add_trace(go.Bar(
    x=season_overview['season'], y=season_overview['Losses'],
    name='Losses', marker_color='#e74c3c', text=season_overview['Losses'], textposition='inside',
), secondary_y=False)
fig_sov.add_trace(go.Scatter(
    x=season_overview['season'], y=season_overview['WinRate'],
    name='Win Rate %', mode='lines+markers+text',
    line=dict(color='#3498db', width=2), marker=dict(size=8),
    text=season_overview['WinRate'].astype(str) + '%', textposition='top center',
), secondary_y=True)
fig_sov.update_layout(
    title=f'Season Overview — {player_label}',
    barmode='stack', legend_title_text='',
)
fig_sov.update_yaxes(title_text='Games', secondary_y=False)
fig_sov.update_yaxes(title_text='Win Rate (%)', range=[0, 110], secondary_y=True)
fig_sov.show()

# ── Per-season top opponents: grouped stacked Win/Loss bars ──
top_n_season = 6

rows = []
for season in seasons_sorted:
    df_s = pm[pm['season'] == season]
    stats_s = matchup_stats(df_s, top_n=top_n_season)
    for rank, (_, row) in enumerate(stats_s.iterrows(), start=1):
        rows.append({
            'season': season,
            'opp_label': row['opp_label'],
            'rank': rank,
            'Wins': int(row['Wins']),
            'Losses': int(row['Losses']),
            'Games': int(row['Games']),
            'WinRate': row['WinRate'],
        })

season_opp_df = pd.DataFrame(rows)

mid = len(seasons_sorted) // 2
season_splits = [seasons_sorted[:mid], seasons_sorted[mid:]]
split_labels = ['(Early Seasons)', '(Late Seasons)']


def make_season_opp_chart(season_subset, subtitle):
    df_sub = season_opp_df[season_opp_df['season'].isin(season_subset)].copy()

    fig = go.Figure()
    wins_in_legend = False
    losses_in_legend = False

    for rank in range(1, top_n_season + 1):
        df_rank = df_sub[df_sub['rank'] == rank].copy()
        if df_rank.empty:
            continue
        og = f'rank{rank}'

        # Wins — bottom segment
        fig.add_trace(go.Bar(
            name='Wins',
            x=df_rank['season'],
            y=df_rank['Wins'],
            marker_color='#2ecc71',
            legendgroup='Wins',
            showlegend=not wins_in_legend,
            offsetgroup=og,
            customdata=df_rank[['opp_label', 'Games', 'WinRate', 'Losses']].values,
            hovertemplate=(
                '<b>%{x}</b> — Rank #' + str(rank) + '<br>'
                '%{customdata[0]}<br>'
                'Wins: %{y}  Losses: %{customdata[3]}<br>'
                'Games: %{customdata[1]}  WR: %{customdata[2]}%<extra></extra>'
            ),
        ))
        wins_in_legend = True

        # Losses — top segment, base=Wins so it sits directly above
        fig.add_trace(go.Bar(
            name='Losses',
            x=df_rank['season'],
            y=df_rank['Losses'],
            base=df_rank['Wins'].tolist(),
            marker_color='#e74c3c',
            legendgroup='Losses',
            showlegend=not losses_in_legend,
            offsetgroup=og,
            text=df_rank['opp_label'],
            textposition='outside',
            textangle=-90,
            textfont=dict(size=11),
            outsidetextfont=dict(size=11),
            constraintext='none',
            cliponaxis=False,
            customdata=df_rank[['opp_label', 'Games', 'WinRate', 'Wins']].values,
            hovertemplate=(
                '<b>%{x}</b> — Rank #' + str(rank) + '<br>'
                '%{customdata[0]}<br>'
                'Wins: %{customdata[3]}  Losses: %{y}<br>'
                'Games: %{customdata[1]}  WR: %{customdata[2]}%<extra></extra>'
            ),
        ))
        losses_in_legend = True

    max_label_len = df_sub['opp_label'].str.len().max() if not df_sub.empty else 10
    # Vertical label (textangle=-90) extends upward; each character needs ~8px of height.
    # Put that space in the top margin so the plot area itself stays compact.
    label_top_margin = max(120, max_label_len * 8)

    fig.update_layout(
        title=f'Top {top_n_season} Opponents per Season — {player_label} {subtitle}',
        barmode='group',
        xaxis=dict(title='Season', categoryorder='array', categoryarray=season_subset),
        yaxis_title='Games',
        legend_title_text='',
        height=420 + label_top_margin,
        margin=dict(t=label_top_margin, b=60, l=60, r=20),
        bargap=0.15,
        bargroupgap=0.05,
    )
    fig.show()


make_season_opp_chart(season_splits[0], split_labels[0])
make_season_opp_chart(season_splits[1], split_labels[1])


In [14]:
# ── CHARACTER MATCHUP HEATMAP ─────────────────────────────────────────────────
# Win rate when player uses character Y vs opponent's character X

def char_heatmap(df, title):
    agg = df.groupby(['player_char_name', 'opp_char_name']).agg(
        Games=('result', 'count'),
        Wins=('result', lambda x: (x == 'Win').sum()),
    ).reset_index()
    agg['WinRate'] = (agg['Wins'] / agg['Games'] * 100).round(1)
    agg['Losses'] = agg['Games'] - agg['Wins']

    wr_matrix = agg.pivot(index='player_char_name', columns='opp_char_name', values='WinRate')
    g_matrix  = agg.pivot(index='player_char_name', columns='opp_char_name', values='Games')
    w_matrix  = agg.pivot(index='player_char_name', columns='opp_char_name', values='Wins').fillna(0)
    l_matrix  = agg.pivot(index='player_char_name', columns='opp_char_name', values='Losses').fillna(0)

    row_wins   = w_matrix.sum(axis=1).astype(int)
    row_losses = l_matrix.sum(axis=1).astype(int)
    col_wins   = w_matrix.sum(axis=0).astype(int)
    col_losses = l_matrix.sum(axis=0).astype(int)

    # Cell text: "WR%\n(N games)"
    text_vals = []
    for r in wr_matrix.index:
        row_text = []
        for c in wr_matrix.columns:
            wr = wr_matrix.loc[r, c]
            g  = g_matrix.loc[r, c] if r in g_matrix.index and c in g_matrix.columns else np.nan
            row_text.append(f"{wr:.0f}%<br>({int(g)})" if pd.notna(wr) else "")
        text_vals.append(row_text)

    fig = make_subplots(
        rows=2, cols=2,
        column_widths=[0.15, 0.85],
        row_heights=[0.85, 0.15],
        shared_xaxes='columns',
        shared_yaxes='rows',
        horizontal_spacing=0.01,
        vertical_spacing=0.01,
    )

    # Main heatmap (top-right)
    fig.add_trace(go.Heatmap(
        z=wr_matrix.values,
        x=list(wr_matrix.columns),
        y=list(wr_matrix.index),
        text=text_vals,
        texttemplate="%{text}",
        colorscale='Blues',
        zmin=0, zmax=100,
        colorbar=dict(title='Win %'),
    ), row=1, col=2)

    # Row totals — stacked horizontal bars on the left (wins at base, losses on top)
    fig.add_trace(go.Bar(
        x=row_wins[wr_matrix.index].values,
        y=list(wr_matrix.index),
        orientation='h',
        marker_color='#2ecc71',
        name='Wins',
        legendgroup='Wins',
        showlegend=False,
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=row_losses[wr_matrix.index].values,
        y=list(wr_matrix.index),
        orientation='h',
        marker_color='#e74c3c',
        name='Losses',
        legendgroup='Losses',
        showlegend=False,
    ), row=1, col=1)

    # Column totals — stacked vertical bars at the bottom (wins at base, losses on top)
    fig.add_trace(go.Bar(
        x=list(wr_matrix.columns),
        y=col_wins[wr_matrix.columns].values,
        marker_color='#2ecc71',
        name='Wins',
        legendgroup='Wins',
        showlegend=False,
    ), row=2, col=2)
    fig.add_trace(go.Bar(
        x=list(wr_matrix.columns),
        y=col_losses[wr_matrix.columns].values,
        marker_color='#e74c3c',
        name='Losses',
        legendgroup='Losses',
        showlegend=False,
    ), row=2, col=2)

    # Left bar: reverse x so bars grow away from heatmap; show tick numbers
    fig.update_xaxes(autorange='reversed', title_text='', showticklabels=True, row=1, col=1)
    fig.update_yaxes(title_text="Player's Character", row=1, col=1)
    fig.update_xaxes(title_text="Opponent's Character", row=2, col=2)
    fig.update_yaxes(title_text='', showticklabels=True, row=2, col=2)

    fig.update_layout(
        title=title,
        barmode='stack',
        height=max(420, len(wr_matrix.index) * 55 + 200),
    )
    fig.show()

# Overall heatmap
char_heatmap(
    pm,
    f"Character Matchup Win Rate — All Seasons ({player_label})"
)

# Per-season heatmaps (skip seasons where the player has no recorded matches)
seasons_sorted_numeric = sorted(seasons, key=lambda s: int(s[1:]))
for season in seasons_sorted_numeric:
    df_s = pm[pm['season'] == season]
    if df_s['player_char_name'].nunique() == 0:
        continue
    char_heatmap(
        df_s,
        f"Character Matchup Win Rate — {season}  ({player_label})"
    )
